In [1]:
"""
PIPELINE STAGE: Multi-Task Energy Data Mining & Longitudinal Extraction
PERIOD: November 2021 - December 2024
LIBRARIES: os, pandas, python-docx

1. OBJECTIVE:
    Automate the discovery and extraction of energy metrics from Word documents across a 4-year 
    period. This stage implements a dual-task architecture to simultaneously process 
    'Subscriber Counts' and 'Electricity Consumption' while ensuring structural validation.

2. TECHNICAL REFINEMENTS (Advanced Data Cleaning):
    A. Intelligent Text Normalization: Beyond simple whitespace stripping, the pipeline 
       replaces newlines (\n) and carriage returns (\r) to prevent row misalignment during 
       DataFrame construction.
    B. Geographic Mapping & Normalization: A dedicated mapping dictionary resolves inconsistent 
       naming conventions (e.g., "K.MARAŞ" -> "KAHRAMANMARAŞ", "ADIYAMAN*" -> "ADIYAMAN") 
       ensuring 100% join-integrity for spatial analysis.
    C. Triple-Check Table Validation: Each detected table is verified by inspecting the 
       first 3 rows for research-specific keywords (İl, Mesken, Sanayi, etc.).

3. MINING STRATEGY:
    A. Aggressive Context Retrieval: Reconstructs table context by merging the 5 preceding 
       paragraphs to resolve header ambiguity.
    B. Positional Data Target: Dynamically targets the first three columns (Province, Category, 
       Value) across multi-task tables to maintain schema uniformity.
    C. Temporal Filtering: Specific logic for the 2021 transition year, processing only 
       late-year reports (November-December).

4. TASK ARCHITECTURE:
    - Task 1 (Subscriber_Count): Extraction of consumer count distributions.
    - Task 2 (Electricity_Consumption): Extraction of billed energy usage patterns.
    - Metadata Injection: Each row is enriched with 'Month', 'Year', and 'Source_Filename'.

5. OUTPUTS (Pipeline Step 04):
    - Final Subscriber Data: ..\processed_data\steps\04_2020_2021_subscriber_1.xlsx
    - Final Consumption Data: ..\processed_data\steps\04_2020_2021_consumption_1.xlsx

6. ERROR HANDLING & LOGGING:
    - Robust exception handling per file ensures the pipeline continues despite corrupt docs.
    - Audit Trail: Real-time console status for every processed year and file.
"""
import os
import pandas as pd
from docx import Document

def veri_temizle(metin):
    """Metindeki gizli karakterleri ve fazla boşlukları temizler."""
    if metin:
        # Satır sonlarını boşluğa çevir, sonra fazla boşlukları temizle
        temiz_metin = " ".join(metin.replace('\n', ' ').replace('\r', ' ').split()).strip()
        return temiz_metin
    return ""

def il_ismini_duzelt(il_adi):
    """Kısa veya hatalı yazılmış il isimlerini normalize eder."""
    if not il_adi:
        return il_adi
    
    # Mapping sözlüğü
    duzeltme_sozlugu = {
        "ADIYAMAN*": "ADIYAMAN",
        "AFYONK.": "AFYONKARAHİSAR",
        "K.MARAŞ": "KAHRAMANMARAŞ",
        "KAHRAMANMARAŞ *": "KAHRAMANMARAŞ",
        "KAHRAMANMARAŞ*": "KAHRAMANMARAŞ"
    }
    
    # Önce tam eşleşme kontrolü, yoksa temizlenmiş metin üzerinden kontrol
    temp_il = il_adi.strip()
    return duzeltme_sozlugu.get(temp_il, temp_il)

def tablo_dogrula(tablo):
    """Tablonun içeriğinde kritik sütunların varlığını kontrol eder."""
    icerik = ""
    for i in range(min(3, len(tablo.rows))):
        try:
            icerik += " ".join([cell.text for cell in tablo.rows[i].cells])
        except: continue
    
    anahtar_kelimeler = ["İl", "Mesken", "Sanayi", "Ticarethane", "Tarımsal"]
    return any(kelime in icerik for kelime in anahtar_kelimeler)

def tabloyu_bul_agresif(doc, hedef_basliklar):
    """Her tabloyu inceler ve üstündeki paragraflarda başlık arar."""
    hedef_basliklar_temiz = [veri_temizle(b) for b in hedef_basliklar]
    
    for tablo in doc.tables:
        p = tablo._element.getprevious()
        sayac = 0
        while p is not None and sayac < 5:
            if p.tag.endswith('p'):
                para_metin = veri_temizle(p.text if hasattr(p, 'text') else "")
                if any(baslik in para_metin for baslik in hedef_basliklar_temiz):
                    if tablo_dogrula(tablo):
                        return tablo
            p = p.getprevious()
            sayac += 1
    return None

def veri_madenciligi_operasyonu():
    # --- YOL DÜZENLEMELERİ ---
    # Kaynak dosyaların bulunduğu ana dizin
    ana_yol = os.path.join("..", "..", "raw_data", "energy_reports") 
    
    yillar = ["2021", "2022", "2023", "2024"]
    
    # Görev tanımları ve çıktı yolları (İki üst dizine çıkacak şekilde)
    gorevler = [
        {
            "ad": "Tuketici_Sayisi",
            "basliklar": [
                "Tüketici Sayısının İl ve Tüketici Türü Bazında Dağılımının Dönemler Arası Karşılaştırılması",
                "Tüketici Sayısının İl Bazında Dağılımının Dönemler Arası Karşılaştırılması"
            ],
            "cikti": os.path.join("..", "..", "processed_data", "steps", "04_2021_2024_subscriber_1.xlsx"),
            "veriler": []
        },
        {
            "ad": "Elektrik_Tuketimi",
            "basliklar": [
                "Faturalanan Elektrik Tüketiminin İl ve Tüketici Türü Bazında Dağılımının Dönemler Arası Karşılaştırılması"
            ],
            "cikti": os.path.join("..", "..", "processed_data", "steps", "04_2021_2024_consumption_1.xlsx"),
            "veriler": []
        }
    ]

    # --- OPERASYON ---
    for yil in yillar:
        yil_dizini = os.path.join(ana_yol, yil)
        if not os.path.exists(yil_dizini): 
            print(f"Dizin bulunamadı, atlanıyor: {yil_dizini}")
            continue

        print(f"\n--- {yil} Yılı Taraması Başladı ---")
        
        for dosya_adi in os.listdir(yil_dizini):
            if not dosya_adi.endswith(".docx"): continue
            
            # 2021 Filtresi: Sadece Kasım (November) ve Aralık (December)
            if yil == "2021" and not any(ay in dosya_adi for ay in ["November", "December"]):
                continue

            try:
                doc = Document(os.path.join(yil_dizini, dosya_adi))
                
                for gorev in gorevler:
                    # Not: tabloyu_bul_agresif fonksiyonunun tanımlı olduğu varsayılmıştır
                    tablo = tabloyu_bul_agresif(doc, gorev["basliklar"])

                    if tablo:
                        tablo_verisi = []
                        for row in tablo.rows:
                            # Not: veri_temizle fonksiyonunun tanımlı olduğu varsayılmıştır
                            satir = [veri_temizle(cell.text) for cell in row.cells]
                            tablo_verisi.append(satir)
                        
                        df_gecici = pd.DataFrame(tablo_verisi)

                        if df_gecici.shape[1] >= 3:
                            # 1, 2 ve 3. sütunlar (0, 1, 2. indexler)
                            df_filtreli = df_gecici.iloc[:, [0, 1, 2]].copy()
                            df_filtreli.columns = ['İl', 'Tüketici Türü', 'Deger']
                            
                            # İL İSİMLERİNİ DÜZELTME
                            # Not: il_ismini_duzelt fonksiyonunun tanımlı olduğu varsayılmıştır
                            df_filtreli['İl'] = df_filtreli['İl'].apply(il_ismini_duzelt)
                            
                            df_filtreli['Ay'] = dosya_adi.split('_')[0]
                            df_filtreli['Yil'] = yil
                            df_filtreli['Kaynak_Dosya'] = dosya_adi
                            
                            gorev["veriler"].append(df_filtreli)
                            print(f"BAŞARILI [{gorev['ad']}]: {dosya_adi}")
            
            except Exception as e:
                print(f"HATA ({dosya_adi}): {e}")

    # --- KAYIT İŞLEMLERİ ---
    for gorev in gorevler:
        if gorev["veriler"]:
            # Çıktı klasörü yoksa oluştur
            os.makedirs(os.path.dirname(gorev["cikti"]), exist_ok=True)
            
            final_df = pd.concat(gorev["veriler"], ignore_index=True)
            final_df.to_excel(gorev["cikti"], index=False)
            print(f"\nTamamlandı: {gorev['ad']} -> {gorev['cikti']}")
        else:
            print(f"\nUyarı: {gorev['ad']} için veri bulunamadı.")

if __name__ == "__main__":
    # Not: veri_temizle, tabloyu_bul_agresif ve il_ismini_duzelt 
    # fonksiyonlarının bu koddan önce tanımlanmış olması gerekir.
    veri_madenciligi_operasyonu()

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\ismailgulsoy\AppData\Local\Temp\ipykernel_13328\4129814463.py:1: SyntaxWarning: invalid escape sequence '\p'
  """



--- 2021 Yılı Taraması Başladı ---
BAŞARILI [Tuketici_Sayisi]: December_2021.docx
BAŞARILI [Elektrik_Tuketimi]: December_2021.docx
BAŞARILI [Tuketici_Sayisi]: November_2021.docx
BAŞARILI [Elektrik_Tuketimi]: November_2021.docx

--- 2022 Yılı Taraması Başladı ---
BAŞARILI [Tuketici_Sayisi]: April_2022.docx
BAŞARILI [Elektrik_Tuketimi]: April_2022.docx
BAŞARILI [Tuketici_Sayisi]: August_2022.docx
BAŞARILI [Elektrik_Tuketimi]: August_2022.docx
BAŞARILI [Tuketici_Sayisi]: December_2022.docx
BAŞARILI [Elektrik_Tuketimi]: December_2022.docx
BAŞARILI [Tuketici_Sayisi]: February_2022.docx
BAŞARILI [Elektrik_Tuketimi]: February_2022.docx
BAŞARILI [Tuketici_Sayisi]: January_2022.docx
BAŞARILI [Elektrik_Tuketimi]: January_2022.docx
BAŞARILI [Tuketici_Sayisi]: July_2022.docx
BAŞARILI [Elektrik_Tuketimi]: July_2022.docx
BAŞARILI [Tuketici_Sayisi]: June_2022.docx
BAŞARILI [Elektrik_Tuketimi]: June_2022.docx
BAŞARILI [Tuketici_Sayisi]: March_2022.docx
BAŞARILI [Elektrik_Tuketimi]: March_2022.docx
BA